Run the following cell if you are in Google Colab. If not, avoid it

In [ ]:
from astropy.table import Table
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import pymc3 as pm
import pymc3_ext as pmx
import exoplanet as xo
import os
import starry
from corner import corner
import theano.tensor as tt
from tqdm.notebook import tqdm
from scipy.stats import median_abs_deviation

import lightkurve as lk

np.random.seed(12)
starry.config.lazy = False
starry.config.quiet = True

In [ ]:
## Variables de nombres:
LC = 'LC_TIC_26474_S06_flatten'#cambiar el nombre segun está en la carpeta de processed
Label = 'TIC_26474_S06_Parsec'

inc_A, sd_inc_A   = 89.5, 0.5  # inclination in degrees
amp_A, sd_amp_A   = 0.87, 0.01 # amplitude (a value prop. to luminosity)
r_A, sd_r_A       = 2.0335, 0.1637 #  radius in R_sun
m_A, sd_m_A       = 1.295, 0.2205  # mass in M_sun
u_A               = [0.3, 0.2],  # limb darkening coefficients

inc_B, sd_inc_B   = 89.5, 0.5  # inclination in degrees
amp_B, sd_amp_B   = 0.13, 0.01 # amplitude (a value prop. to luminosity)
r_B, sd_r_B       = 1.2656, 0.1017  #  radius in R_sun
m_B, sd_m_B       = 0.2605, 0.1472  #  mass in M_sun
porb_B, sd_porb_B = 3.038, 0.01  # orbital period in days
t0_B, sd_t0_B     = 1469.38, 0.01  # reference time in days (when it transits A)
ecc_B, sd_ecc_B   = 0.0, 0.05
w_B, sd_w_B       = 90, 5  # longitude of pericenter in degrees
u_B               = [0.3, 0.2],  # limb darkening coefficients

prot_s            = 0

In [ ]:
lightcurve = Table.read(LC, format = 'ascii.csv')

t_real = lightcurve['time']
flux_real = lightcurve['flux']
sigma_real = lightcurve['flux_err']
sigma_sd = 1.4826*median_abs_deviation(flux_real)

In [ ]:
fig, ax = plt.subplots(1, figsize=(12,5))
ax.plot(t_real, flux_real, ".k", alpha=0.6, ms=5)
ax.set_xlabel("time [days]", fontsize=24)
ax.set_ylabel("normalized flux", fontsize=24);

In [ ]:
map = starry.Map(ydeg=5)
A_y = np.array(map.y[1:])

map.reset()
B_y = np.array(map.y[1:])

In [ ]:
A = dict(
    ydeg   = 5,  # degree of the map
    udeg   = 2,  # degree of the limb darkening
    inc    = inc_A,  # inclination in degrees
    amp    = amp_A,  # amplitude (a value prop. to luminosity)
    r      = r_A, #  radius in R_sun
    m      = m_A,  # mass in M_sun
    prot   = prot_s,  # rotational period in days
    u      = u_A[0],  # limb darkening coefficients
    y      = A_y,  # the spherical harmonic coefficients
)
B = dict(
    ydeg   = 5,  # degree of the map
    udeg   = 2,  # degree of the limb darkening
    inc    = inc_B,  # inclination in degrees
    amp    = amp_B,  # amplitude (a value prop. to luminosity)
    r      = r_B,  #  radius in R_sun
    m      = m_B,  #  mass in M_sun
    prot   = prot_s,  # rotational period in days
    porb   = porb_B,  # orbital period in days
    t0     = t0_B,  # reference time in days (when it transits A)
    ecc    = ecc_B,
    w      = w_B,  # longitude of pericenter in degrees
    u      = u_B[0],  # limb darkening coefficients
    y      = B_y,  # the spherical harmonic coefficients
)

In [ ]:
pri = starry.Primary(
    starry.Map(ydeg=A["ydeg"], udeg=A["udeg"], inc=A["inc"], amp=A["amp"]),
    r=A["r"],
    m=A["m"],
    prot=A["prot"],
)
pri.map[1:] = A["u"]
pri.map[1:, :] = A["y"]

sec = starry.Secondary(
    starry.Map(ydeg=B["ydeg"], udeg=B["udeg"], inc=B["inc"], amp=B["amp"]),
    r=B["r"],
    m=B["m"],
    porb=B["porb"],
    prot=B["prot"],
    t0=B["t0"],
    inc=B["inc"],
    ecc=B["ecc"],
    w=B['w'],
)
sec.map[1:] = B["u"]
sec.map[1:, :] = B["y"]

In [ ]:
sys = starry.System(pri, sec)

In [ ]:
t = t_real
flux_true = sys.flux(t_real)
sigma = sigma_real
# flux = flux_real + sigma_real * np.random.randn(len(t))
flux = flux_real + sigma_sd * np.random.randn(len(t))

In [ ]:
fig, ax = plt.subplots(1, figsize=(18, 9))
ax.plot(t_real, flux_real, "k.", alpha=0.5, ms=10)
ax.plot(t_real, flux_true, lw=2)
# ax.plot(t_real, flux_real, ".k", ms=4, color= 'k' ,label="Sector 06")
ax.set_xlabel("time [days]", fontsize=24)
ax.set_ylabel("normalized flux", fontsize=24);

#ax.set_xlim(1469 , 1470);

In [ ]:
np.savez(Label+".npz", A=A, B=B, t=t_real, flux=flux, sigma=sigma_real)

np.random.seed(12)
starry.config.lazy = True
starry.config.quiet = True

data = np.load(Label+".npz", allow_pickle=True)
A = data["A"].item()
B = data["B"].item()
t = data["t"]
flux = data["flux"]
sigma = data["sigma"]

In [ ]:
fig, ax = plt.subplots(1, figsize=(12, 5))
ax.plot(t, flux_real, "b.", alpha=0.5, ms=5)
ax.plot(t, flux, "k.", alpha=0.5, ms=5)
ax.set_xlabel("time [days]", fontsize=24)
ax.set_ylabel("normalized flux", fontsize=24);

In [ ]:
with pm.Model() as model:
    PositiveNormal = pm.Bound(pm.Normal, lower=0.0)

    # Primary
    A_inc = pm.Normal("A_inc", mu = inc_A, sd = sd_inc_A, testval = inc_A)
    A_amp = pm.Normal("A_amp", mu = amp_A, sd = sd_amp_A, testval= amp_A)
    A_r = PositiveNormal("A_r", mu = r_A, sd = sd_r_A, testval = r_A)
    A_m = PositiveNormal("A_m", mu = m_A, sd = sd_m_A, testval = m_A)
    A_prot = prot_s
    u_A = xo.distributions.QuadLimbDark("u_A", testval=np.array(A['u']))
    pri = starry.Primary(
        starry.Map(ydeg=A["ydeg"], udeg=A["udeg"], inc=A_inc, amp=A_amp),
        r=A_r,
        m=A_m,
        prot=A_prot,
    )
    pri.map[1] = A["u"][0]
    pri.map[2] = A["u"][1]

    # Secondary
    B_inc = pm.Normal("B_inc", mu = inc_B, sd = sd_inc_B, testval = inc_B)
    B_amp = pm.Normal("B_amp", mu = amp_B, sd = sd_amp_B, testval = amp_B)
    B_r = PositiveNormal("B_r", mu = r_B, sd = sd_r_B, testval = r_B)
    B_m = PositiveNormal("B_m", mu = m_B, sd = sd_m_B, testval = m_B)
    B_prot = prot_s
    B_porb = PositiveNormal("B_porb", mu = porb_B, sd = sd_porb_B, testval = porb_B)
    B_t0 = pm.Normal("B_t0", mu = t0_B, sd = sd_t0_B, testval = t0_B)
    u_B = xo.distributions.QuadLimbDark("u_B", testval=np.array(B['u']))
    B_ecc = pm.TruncatedNormal("B_ecc", mu = ecc_B, sigma = sd_ecc_B, lower = 0, upper = 1)
    B_w = pm.Normal("B_w", mu = w_B, sd = sd_w_B, testval = w_B)#pm.Uniform("B_w", 0, 360, initval = w_B)
    #B_Omega = pm.Uniform("B_Omega", 0, 360, initval=Omega_B)
    sec = starry.Secondary(
        starry.Map(ydeg=B["ydeg"], udeg=B["udeg"], inc=B_inc, amp=B_amp),
        r=B_r,
        m=B_m,
        porb=B_porb,
        prot=B_prot,
        t0=B_t0,
        inc=B_inc,
        ecc=B_ecc,
        w=B_w,
        #Omega=B_Omega
    )
    sec.map[1] = B["u"][0]
    sec.map[2] = B["u"][1]

    # System
    sys = starry.System(pri, sec)

    # Compute the flux
    flux_model = sys.flux(t=t)

    # Track some values for plotting later
    pm.Deterministic("flux_model", flux_model)

    # Save our initial guess
    flux_model_guess = pmx.eval_in_model(flux_model)

    # The likelihood function assuming known Gaussian uncertainty
    pm.Normal("obs", mu=flux_model, sd=sigma_sd, observed=flux_real)


In [ ]:
with model:
    map_soln = pmx.optimize(start=model.test_point)

In [ ]:
map_soln

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(t, flux, "k.", alpha=0.3, ms=2, label="data")
plt.plot(t, flux_model_guess, "C1--", lw=1, alpha=0.5, label="Initial")
plt.plot(
    t, pmx.eval_in_model(flux_model, map_soln, model=model), "C1-", label="MAP", lw=1
)
plt.legend(fontsize=10, numpoints=5)
plt.xlabel("time [days]", fontsize=24)
plt.ylabel("relative flux", fontsize=24);

In [ ]:
with model:
    trace = pmx.sample(
        tune=1500, draws=1500, start=map_soln, cores=4, chains=4, target_accept=0.9, return_inferencedata=True,
    )

In [ ]:
var_names_A = ["A_m", "A_r", "A_inc", "u_A", 'A_amp']
var_names_B = ["B_m", "B_r", "B_porb", "B_inc", "B_t0", "B_ecc", "B_w", 'u_B', 'B_amp']
display(pm.summary(trace, var_names=var_names_A + var_names_B))

# Obtener el resumen
summary = pm.summary(trace, var_names=var_names_A + var_names_B)

# Guardar en un archivo CSV
summary.to_csv(Label+'_Summary.csv', sep=',')

In [ ]:
figure = corner(trace, var_names=var_names_A, show_titles=True,use_math_text=True,label_kwargs={"fontsize": 14})

for ax in figure.get_axes():
      ax.tick_params(axis='both', labelsize=12)


figure.savefig(Label+'_A.png',dpi=600,pad_inches=0.3,bbox_inches='tight')

In [ ]:
figure = corner(trace, var_names=var_names_B, show_titles=True,use_math_text=True,label_kwargs={"fontsize": 14})

for ax in figure.get_axes():
      ax.tick_params(axis='both', labelsize=12)


figure.savefig(Label+'_B.png',dpi=600, pad_inches=0.3, bbox_inches='tight')

In [ ]:
# Plot the results
curve = trace.posterior["flux_model"].values
q16, q50, q84 = np.percentile(curve, [16, 50, 84], axis=(0, 1))

Curva_Q50 = lk.LightCurve(time=t, flux=q50)
Curva_Q16 = lk.LightCurve(time=t, flux=q16)
Curva_Q84 = lk.LightCurve(time=t, flux=q84)

strm_1 = Curva_Q50.to_csv()
with open(Label+"_q50.csv", "w") as text_file:
    text_file.write(strm_1)

strm_2 = Curva_Q16.to_csv()
with open(Label+"_q16.csv", "w") as text_file:
    text_file.write(strm_2)

strm_3 = Curva_Q84.to_csv()
with open(Label+"_q84.csv", "w") as text_file:
    text_file.write(strm_2)

Curva_Q50

In [ ]:
fig, ax = plt.subplots(1, figsize=(18, 9))
ax.plot(t_real, flux_real, "k.", alpha=0.5, ms=10)
ax.plot(Curva_Q50['time'].value, Curva_Q50['flux'].value, lw=2)
# ax.plot(t_real, flux_real, ".k", ms=4, color= 'k' ,label="Sector 06")
ax.set_xlabel("time [days]", fontsize=24)
ax.set_ylabel("normalized flux", fontsize=24);

In [ ]:
import pandas as pd

inf_pos = {"A_m": trace.posterior["A_m"].values.reshape(-1), "A_r": trace.posterior["A_r"].values.reshape(-1),
           "A_inc": trace.posterior["A_inc"].values.reshape(-1), "A_amp": trace.posterior["A_amp"].values.reshape(-1),
           'A_u0':trace.posterior["u_A"][:,:,0].values.reshape(-1), 'A_u1':trace.posterior["u_A"][:,:,1].values.reshape(-1),
           "B_m": trace.posterior["B_m"].values.reshape(-1), "B_r": trace.posterior["B_r"].values.reshape(-1),
           "B_porb": trace.posterior["B_porb"].values.reshape(-1), "B_inc": trace.posterior["B_inc"].values.reshape(-1),
           "B_t0": trace.posterior["B_t0"].values.reshape(-1), "B_ecc": trace.posterior["B_ecc"].values.reshape(-1),
           "B_w": trace.posterior["B_w"].values.reshape(-1), "B_amp": trace.posterior["B_amp"].values.reshape(-1),
           'B_u0':trace.posterior["u_B"][:,:,0].values.reshape(-1), 'B_u1':trace.posterior["u_B"][:,:,1].values.reshape(-1),}
print(inf_pos["A_m"].shape)
df_pos = pd.DataFrame(inf_pos)

df_pos.to_csv(Label+"_Posteriors.csv", sep=',')

# Calcular el promedio de cada columna y mostrarlo en pantalla
column_means = df_pos.mean()
print(column_means)

In [ ]:
df_pos